# 안구건조증 전안부 영상 각막 Binary Segmentation

MobileNetV2 인코더 기반 U-Net을 이용하여 전안부 형광검사 영상에서 각막(cornea) 영역을 분리하는 이진 분할(binary segmentation) 모델을 훈련합니다.

**수정 사항 요약:**
1. 3-class → Binary segmentation (출력 채널 1, Sigmoid + BinaryCrossentropy)
2. 다양한 데이터 증강 적용 (flip, rotation, brightness/contrast)
3. 입력 해상도 실험을 통한 최적 크기 결정
4. IoU(Intersection over Union) 평가 지표 추가

## 1. 환경 설정

In [ ]:
# 필요 패키지 설치
!pip install -q git+https://github.com/tensorflow/examples.git
!pip install -q scikit-learn

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import glob
from sklearn.model_selection import train_test_split
from tensorflow_examples.models.pix2pix import pix2pix
from IPython.display import clear_output

print(f"TensorFlow 버전: {tf.__version__}")
print(f"GPU 사용 여부: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ─────────────────────────── 하이퍼파라미터 설정 ───────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/Dry_eye_binary_segmentation'
IMG_SIZE    = 256      # 입력 해상도 실험 후 최종 결정값 (128 / 256 / 384)
BATCH_SIZE  = 8        # GPU VRAM에 따라 조정 (T4: 8~16 권장)
EPOCHS      = 60
SEED        = 42
LR          = 1e-4     # Adam optimizer learning rate

tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. 데이터 탐색

데이터 폴더 구조를 자동으로 감지합니다.

지원하는 폴더 구조:
```
Dry_eye_binary_segmentation/
├── images/   ← 원본 영상 (*.png / *.jpg)
└── masks/    ← 이진 마스크 (*.png, 픽셀 값 0 또는 255)
```

In [ ]:
def find_data_paths(root):
    """이미지/마스크 경로를 자동 탐색하여 정렬된 리스트로 반환"""
    exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.tiff')

    # 후보 폴더 패턴
    img_dirs  = ['images', 'image', 'imgs', 'img', 'JPEGImages']
    mask_dirs = ['masks', 'mask', 'annotations', 'labels', 'SegmentationClass']

    img_paths, mask_paths = [], []

    for d in img_dirs:
        folder = os.path.join(root, d)
        if os.path.isdir(folder):
            for ext in exts:
                img_paths += glob.glob(os.path.join(folder, ext))
            if img_paths:
                break

    for d in mask_dirs:
        folder = os.path.join(root, d)
        if os.path.isdir(folder):
            for ext in exts:
                mask_paths += glob.glob(os.path.join(folder, ext))
            if mask_paths:
                break

    img_paths  = sorted(img_paths)
    mask_paths = sorted(mask_paths)

    assert len(img_paths) > 0,  f"이미지를 찾지 못했습니다: {root}"
    assert len(img_paths) == len(mask_paths), \
        f"이미지({len(img_paths)})와 마스크({len(mask_paths)}) 수가 다릅니다!"

    return img_paths, mask_paths


image_paths, mask_paths = find_data_paths(DRIVE_ROOT)
print(f"전체 이미지 수: {len(image_paths)}")
print(f"이미지 경로 예시: {image_paths[0]}")
print(f"마스크 경로 예시: {mask_paths[0]}")

In [ ]:
# Train / Validation 분리  (8:2)
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=SEED
)
print(f"훈련 샘플: {len(train_imgs)},  검증 샘플: {len(val_imgs)}")

In [ ]:
# ─────────────────────── Figure 1: 원본 데이터 샘플 시각화 ───────────────────────
# 보고서 위치: 2장 '데이터' 섹션 맨 앞

def read_raw(img_path, mask_path):
    img = plt.imread(img_path)
    msk = plt.imread(mask_path)
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    if img.max() > 1:
        img = img / 255.0
    msk = msk if msk.ndim == 2 else msk[:, :, 0]
    msk = (msk > 0.5).astype(np.float32)
    return img, msk

n_show = min(4, len(image_paths))
fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
fig.suptitle('Figure 1: Raw Data Samples (Top: Input Image, Bottom: Ground-Truth Mask)',
             fontsize=13, fontweight='bold')

for i in range(n_show):
    img, msk = read_raw(image_paths[i], mask_paths[i])
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')
    axes[1, i].imshow(msk, cmap='gray')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('Figure_1_Data_Samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: Figure_1_Data_Samples.png")

## 3. 데이터 전처리 및 증강

### Binary Segmentation 전환
- 원본 코드: `OUTPUT_CLASSES = 3`, Softmax, SparseCategoricalCrossentropy
- 수정 후: `OUTPUT_CLASSES = 1`, Sigmoid, BinaryCrossentropy
- 마스크: 픽셀값을 0.0 / 1.0 (이진) 으로 정규화

### 증강 기법 선택 근거
| 기법 | 선택 이유 |
|------|----------|
| Random Horizontal/Vertical Flip | 각막은 좌우/상하 대칭 구조, 방향 불변성 학습 |
| Random Rotation (±15°) | 촬영 시 눈 기울기 변동 반영 |
| Random Brightness/Contrast | 형광 조명 강도 변화, 카메라 노출 차이 보정 |
| Random Crop & Resize | 미세 이동 변화 대응, 실효 데이터 다양성 향상 |

In [ ]:
# ──────────────────────── 이미지/마스크 로딩 함수 ────────────────────────

def load_image_mask(img_path, mask_path, img_size=IMG_SIZE):
    # 이미지 로드 및 리사이즈
    img = tf.io.read_file(img_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_size, img_size])
    img = tf.cast(img, tf.float32) / 255.0

    # 마스크 로드 및 이진화
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_image(mask, channels=1, expand_animations=False)
    mask = tf.image.resize(mask, [img_size, img_size], method='nearest')
    mask = tf.cast(mask, tf.float32) / 255.0
    mask = tf.where(mask > 0.5, 1.0, 0.0)  # 완전 이진화

    return img, mask

In [ ]:
# ──────────────────────── 데이터 증강 함수 ────────────────────────
# 공간적 변환: 이미지+마스크를 함께 처리 (채널 축 concat 기법)
# 색상 변환: 이미지에만 적용

@tf.function
def augment(image, mask):
    # ── 공간 변환: image(3ch) + mask(1ch) 를 concat해 동일 변환 적용 ──
    combined = tf.concat([image, mask], axis=-1)  # (H, W, 4)

    # Random Horizontal Flip
    combined = tf.image.random_flip_left_right(combined, seed=SEED)

    # Random Vertical Flip
    combined = tf.image.random_flip_up_down(combined, seed=SEED)

    # Random Rotation ±15° — 채널 4짜리 tensor에 적용
    angle = tf.random.uniform((), minval=-0.2618, maxval=0.2618)  # ±15도 (라디안)
    combined = _rotate_tensor(combined, angle)

    # Random Crop & Resize (±15%)
    h, w = tf.shape(combined)[0], tf.shape(combined)[1]
    crop_frac = tf.random.uniform((), 0.85, 1.0)
    crop_h = tf.cast(tf.cast(h, tf.float32) * crop_frac, tf.int32)
    crop_w = tf.cast(tf.cast(w, tf.float32) * crop_frac, tf.int32)
    combined = tf.image.random_crop(
        combined, size=[crop_h, crop_w, 4], seed=SEED
    )
    combined = tf.image.resize(combined, [IMG_SIZE, IMG_SIZE])

    # 채널 분리
    image = combined[:, :, :3]
    mask  = combined[:, :, 3:]
    mask  = tf.where(mask > 0.5, 1.0, 0.0)  # 리사이즈 후 재이진화

    # ── 색상 변환: 이미지에만 적용 ──
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.75, upper=1.25)
    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, mask


def _rotate_tensor(tensor, angle):
    """중심 기준 회전 변환 (ProjectiveTransformV3 이용)"""
    cos_angle = tf.math.cos(angle)
    sin_angle = tf.math.sin(angle)
    h = tf.cast(tf.shape(tensor)[0], tf.float32)
    w = tf.cast(tf.shape(tensor)[1], tf.float32)
    cx, cy = w / 2.0, h / 2.0

    a0 =  cos_angle
    a1 = -sin_angle
    a2 = cx - cx * cos_angle + cy * sin_angle
    b0 =  sin_angle
    b1 =  cos_angle
    b2 = cy - cx * sin_angle - cy * cos_angle
    transform = tf.reshape(tf.stack([a0, a1, a2, b0, b1, b2, 0.0, 0.0]), [1, 8])

    tensor = tf.expand_dims(tensor, 0)  # (1, H, W, C)
    tensor = tf.raw_ops.ImageProjectiveTransformV3(
        images=tensor,
        transforms=transform,
        output_shape=tf.shape(tensor)[1:3],
        interpolation='BILINEAR',
        fill_mode='REFLECT',
        fill_value=0.0          # ← 누락된 필수 인수 추가
    )
    tensor = tf.squeeze(tensor, 0)  # (H, W, C)
    return tensor

In [ ]:
# ─────────────── Figure 2: 증강 예시 시각화 ───────────────
# 보고서 위치: '데이터 증강 기법 선택 근거' 섹션

sample_img, sample_mask = load_image_mask(train_imgs[0], train_masks[0])

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Figure 2: Data Augmentation Examples', fontsize=13, fontweight='bold')

# 원본
axes[0, 0].imshow(sample_img.numpy())
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')
axes[1, 0].imshow(sample_mask.numpy().squeeze(), cmap='gray')
axes[1, 0].set_title('Original Mask')
axes[1, 0].axis('off')

# 4가지 증강 샘플
for i in range(1, 5):
    aug_img, aug_mask = augment(sample_img, sample_mask)
    axes[0, i].imshow(aug_img.numpy())
    axes[0, i].set_title(f'Augmented {i}')
    axes[0, i].axis('off')
    axes[1, i].imshow(aug_mask.numpy().squeeze(), cmap='gray')
    axes[1, i].set_title(f'Augmented Mask {i}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('Figure_2_Augmentation_Examples.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: Figure_2_Augmentation_Examples.png")

## 4. 입력 해상도 실험

VRAM 부족 없이 각막 형상(morphology)을 보존하는 최적 입력 해상도를 실험적으로 결정합니다.

**실험 전략:**
- 후보 해상도: 128×128, 256×256, 384×384
- 각 해상도에서 각막 경계 보존 여부를 육안 + 마스크 면적 비율로 평가
- VRAM 한계: T4(16GB) 기준 384×384 + Batch 8 ≈ ~11GB (사용 가능)

In [ ]:
# ─────────────────── Figure 3: 입력 해상도 비교 ───────────────────
# 보고서 위치: '입력 해상도 결정 과정' 섹션

resolutions = [128, 256, 384]
raw_img = tf.io.read_file(image_paths[0])
raw_img = tf.image.decode_image(raw_img, channels=3, expand_animations=False)
raw_img = tf.cast(raw_img, tf.float32) / 255.0

raw_mask = tf.io.read_file(mask_paths[0])
raw_mask = tf.image.decode_image(raw_mask, channels=1, expand_animations=False)
raw_mask = tf.cast(raw_mask, tf.float32) / 255.0
raw_mask = tf.where(raw_mask > 0.5, 1.0, 0.0)

orig_area_ratio = tf.reduce_mean(raw_mask).numpy()

fig, axes = plt.subplots(3, len(resolutions), figsize=(5 * len(resolutions), 12))
fig.suptitle('Figure 3: Input Resolution Comparison\n(Row 1: Resized Image, Row 2: Resized Mask, Row 3: Overlay)',
             fontsize=12, fontweight='bold')

for col, res in enumerate(resolutions):
    res_img  = tf.image.resize(raw_img,  [res, res]).numpy()
    res_mask = tf.image.resize(raw_mask, [res, res], method='nearest')
    res_mask = tf.where(res_mask > 0.5, 1.0, 0.0).numpy().squeeze()

    area_ratio = res_mask.mean()
    change_pct = abs(area_ratio - orig_area_ratio) / (orig_area_ratio + 1e-7) * 100

    axes[0, col].imshow(res_img)
    axes[0, col].set_title(f'{res}×{res}\nImage')
    axes[0, col].axis('off')

    axes[1, col].imshow(res_mask, cmap='gray')
    axes[1, col].set_title(f'Mask\nArea ratio: {area_ratio:.3f}\nChange: {change_pct:.1f}%')
    axes[1, col].axis('off')

    overlay = res_img.copy()
    overlay[res_mask > 0.5] = overlay[res_mask > 0.5] * 0.5 + np.array([0, 1, 0]) * 0.5
    axes[2, col].imshow(overlay)
    axes[2, col].set_title('Overlay')
    axes[2, col].axis('off')

plt.tight_layout()
plt.savefig('Figure_3_Resolution_Comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: Figure_3_Resolution_Comparison.png")
print(f"\n원본 마스크 면적 비율: {orig_area_ratio:.4f}")
print(f"→ 권장 해상도: {IMG_SIZE}×{IMG_SIZE} (형상 보존 + VRAM 효율 균형)")

## 5. tf.data 파이프라인 구성

In [ ]:
def make_dataset(img_paths, mask_paths, img_size=IMG_SIZE,
                 do_augment=False, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((img_paths, mask_paths))
    if shuffle:
        ds = ds.shuffle(len(img_paths), seed=SEED)
    ds = ds.map(
        lambda x, y: load_image_mask(x, y, img_size),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    if do_augment:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_dataset = make_dataset(train_imgs, train_masks,
                             do_augment=True, shuffle=True)
val_dataset   = make_dataset(val_imgs,   val_masks,
                             do_augment=False, shuffle=False)

print(f"훈련 배치 수: {len(train_dataset)}")
print(f"검증 배치 수: {len(val_dataset)}")

# 배치 크기 확인
for imgs, masks in train_dataset.take(1):
    print(f"이미지 배치 shape: {imgs.shape}")
    print(f"마스크 배치 shape: {masks.shape}")

## 6. U-Net 모델 정의 (Binary Segmentation)

### 원본 코드 vs 수정 코드

| 항목 | 원본 (3-class) | 수정 (Binary) |
|------|--------------|---------------|
| 출력 채널 수 | 3 | 1 |
| 마지막 활성화 | (없음, logits) | Sigmoid |
| 손실 함수 | SparseCategoricalCrossentropy | BinaryCrossentropy |
| 마스크 예측 | argmax → 0/1/2 | threshold 0.5 → 0/1 |
| 평가 지표 | accuracy | accuracy + BinaryIoU |

In [ ]:
# ──────────────── MobileNetV2 인코더 (다운샘플 스택) ────────────────
base_model = tf.keras.applications.MobileNetV2(
    input_shape=[IMG_SIZE, IMG_SIZE, 3], include_top=False
)

layer_names = [
    'block_1_expand_relu',   # IMG_SIZE/2
    'block_3_expand_relu',   # IMG_SIZE/4
    'block_6_expand_relu',   # IMG_SIZE/8
    'block_13_expand_relu',  # IMG_SIZE/16
    'block_16_project',      # IMG_SIZE/32
]
base_model_outputs = [base_model.get_layer(name).output for name in layer_names]
down_stack = tf.keras.Model(inputs=base_model.input, outputs=base_model_outputs)
down_stack.trainable = False  # 인코더 동결 (전이학습)

# ──────────────── 디코더 (업샘플 스택) ────────────────
up_stack = [
    pix2pix.upsample(512, 3),
    pix2pix.upsample(256, 3),
    pix2pix.upsample(128, 3),
    pix2pix.upsample(64,  3),
]


def unet_model_binary():
    """
    Binary Segmentation용 U-Net
    - 출력: (H, W, 1)  sigmoid 활성화
    - 원본과 달리 마지막 레이어에 sigmoid 추가
    """
    inputs = tf.keras.layers.Input(shape=[IMG_SIZE, IMG_SIZE, 3])

    # 다운샘플링 (skip connections 포함)
    skips = down_stack(inputs)
    x = skips[-1]
    skips = reversed(skips[:-1])

    # 업샘플링 + skip connection 연결
    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = tf.keras.layers.Concatenate()([x, skip])

    # 최종 출력 레이어 — Binary: 채널 1, sigmoid
    last = tf.keras.layers.Conv2DTranspose(
        filters=1, kernel_size=3, strides=2, padding='same',
        activation='sigmoid'   # ← 원본의 logits 방식에서 변경
    )
    x = last(x)

    return tf.keras.Model(inputs=inputs, outputs=x)


model = unet_model_binary()
print(f"출력 shape: {model.output_shape}")

In [ ]:
# ─────────────────── Figure 4: 모델 아키텍처 ───────────────────
# 보고서 위치: '모델 구조' 섹션

arch_img = tf.keras.utils.plot_model(
    model, show_shapes=True,
    to_file='Figure_4_Model_Architecture.png',
    dpi=80
)
print("저장 완료: Figure_4_Model_Architecture.png")
arch_img

In [ ]:
# ──────────────── 사용자 정의 Dice Coefficient 지표 ────────────────
class DiceCoefficient(tf.keras.metrics.Metric):
    """Dice = 2*|A∩B| / (|A|+|B|) — binary segmentation 성능 지표"""

    def __init__(self, threshold=0.5, name='dice', **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = threshold
        self.dice_sum  = self.add_weight(name='dice_sum',  initializer='zeros')
        self.count     = self.add_weight(name='count',     initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        intersection = tf.reduce_sum(y_true * y_pred)
        union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
        dice  = (2.0 * intersection + 1e-7) / (union + 1e-7)
        self.dice_sum.assign_add(dice)
        self.count.assign_add(1.0)

    def result(self):
        return self.dice_sum / self.count

    def reset_state(self):
        self.dice_sum.assign(0.0)
        self.count.assign(0.0)


# ──────────────── 모델 컴파일 ────────────────
# 원본: SparseCategoricalCrossentropy(from_logits=True)
# 수정: BinaryCrossentropy (sigmoid 출력이므로 from_logits=False)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        'accuracy',
        tf.keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name='iou'),
        DiceCoefficient(name='dice'),
    ]
)

model.summary()

## 7. 모델 훈련

In [ ]:
# ──────────────── 예측 마스크 생성 (binary threshold) ────────────────
def create_binary_mask(pred_mask, threshold=0.5):
    """sigmoid 출력을 0/1 이진 마스크로 변환 (원본의 argmax 대체)"""
    return tf.where(pred_mask[0] > threshold, 1.0, 0.0)


def display(display_list, title_list=None):
    n = len(display_list)
    if title_list is None:
        title_list = ['Input Image', 'True Mask', 'Predicted Mask', 'Overlay'][:n]
    plt.figure(figsize=(5 * n, 5))
    for i, item in enumerate(display_list):
        plt.subplot(1, n, i + 1)
        plt.title(title_list[i])
        arr = item.numpy() if hasattr(item, 'numpy') else item
        if arr.ndim == 3 and arr.shape[-1] == 1:
            plt.imshow(arr.squeeze(), cmap='gray')
        else:
            plt.imshow(arr)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


def show_predictions(dataset=None, num=1):
    if dataset is not None:
        for images, masks in dataset.take(num):
            pred = model.predict(images, verbose=0)
            display([images[0], masks[0], create_binary_mask(pred)])
    else:
        for images, masks in val_dataset.take(1):
            pred = model.predict(images[:1], verbose=0)
            display([images[0], masks[0], create_binary_mask(pred)])

In [ ]:
# 학습 전 예측 확인 (baseline)
print("[학습 전 예측 — 무작위 가중치]")
show_predictions()

In [ ]:
# ──────────────── 콜백 정의 ────────────────

class DisplayCallback(tf.keras.callbacks.Callback):
    """에포크 종료 시 검증 샘플 예측 출력"""
    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)
        show_predictions()
        print(f'\n── Epoch {epoch+1}/{EPOCHS} ──')
        for k, v in logs.items():
            print(f'  {k}: {v:.4f}')


# ReduceLROnPlateau: val_loss 개선 없으면 학습률 감소
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=8,
    min_lr=1e-7, verbose=1
)

# EarlyStopping: 과적합 방지
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_dice', mode='max',
    patience=15, restore_best_weights=True, verbose=1
)

# 모델 저장
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(DRIVE_ROOT, 'best_model.h5'),
    monitor='val_dice', mode='max',
    save_best_only=True, verbose=1
)

In [ ]:
# ──────────────── 모델 훈련 ────────────────
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=[DisplayCallback(), reduce_lr, early_stop, checkpoint]
)

## 8. 학습 결과 분석

In [ ]:
# ─────────────── Figure 5: 학습 곡선 (Loss & Metrics) ───────────────
# 보고서 위치: '학습 곡선' 섹션

hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Figure 5: Training History', fontsize=13, fontweight='bold')

# Loss
axes[0].plot(epochs_ran, hist['loss'],     'b-o',  markersize=3, label='Train Loss')
axes[0].plot(epochs_ran, hist['val_loss'], 'r--o', markersize=3, label='Val Loss')
axes[0].set_title('Binary Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(epochs_ran, hist['iou'],     'b-o',  markersize=3, label='Train IoU')
axes[1].plot(epochs_ran, hist['val_iou'], 'r--o', markersize=3, label='Val IoU')
axes[1].set_title('Binary IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].set_ylim([0, 1])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Dice
axes[2].plot(epochs_ran, hist['dice'],     'b-o',  markersize=3, label='Train Dice')
axes[2].plot(epochs_ran, hist['val_dice'], 'r--o', markersize=3, label='Val Dice')
axes[2].set_title('Dice Coefficient')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Dice')
axes[2].set_ylim([0, 1])
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Figure_5_Training_History.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: Figure_5_Training_History.png")

best_epoch = np.argmax(hist['val_dice'])
print(f"\n최고 성능 에포크: {best_epoch+1}")
print(f"  Val Loss : {hist['val_loss'][best_epoch]:.4f}")
print(f"  Val IoU  : {hist['val_iou'][best_epoch]:.4f}")
print(f"  Val Dice : {hist['val_dice'][best_epoch]:.4f}")

In [ ]:
# ─────────── Figure 6: 대표 분할 결과 (입력 + 마스크 + 예측 + Overlay) ───────────
# 보고서 위치: '대표 결과 이미지' 섹션

n_samples = min(4, len(val_imgs))
fig, axes = plt.subplots(n_samples, 4,
                          figsize=(16, 4 * n_samples))
fig.suptitle(
    'Figure 6: Segmentation Results\n'
    '(Col 1: Input | Col 2: Ground Truth | Col 3: Prediction | Col 4: Overlay)',
    fontsize=12, fontweight='bold'
)

col_titles = ['Input Image', 'Ground Truth', 'Predicted Mask', 'Overlay']
for col, t in enumerate(col_titles):
    axes[0, col].set_title(t, fontsize=11, fontweight='bold')

for row in range(n_samples):
    img, mask = load_image_mask(val_imgs[row], val_masks[row])
    pred = model.predict(img[tf.newaxis, ...], verbose=0)
    pred_mask = create_binary_mask(pred).numpy().squeeze()

    img_np  = img.numpy()
    mask_np = mask.numpy().squeeze()

    # Overlay: 초록 = 정확 예측, 빨강 = False Positive, 파랑 = False Negative
    overlay = img_np.copy()
    tp = (pred_mask == 1) & (mask_np == 1)
    fp = (pred_mask == 1) & (mask_np == 0)
    fn = (pred_mask == 0) & (mask_np == 1)
    overlay[tp] = overlay[tp] * 0.4 + np.array([0.0, 1.0, 0.0]) * 0.6  # 초록
    overlay[fp] = overlay[fp] * 0.4 + np.array([1.0, 0.0, 0.0]) * 0.6  # 빨강
    overlay[fn] = overlay[fn] * 0.4 + np.array([0.0, 0.0, 1.0]) * 0.6  # 파랑
    overlay = np.clip(overlay, 0, 1)

    axes[row, 0].imshow(img_np)
    axes[row, 1].imshow(mask_np, cmap='gray')
    axes[row, 2].imshow(pred_mask, cmap='gray')
    axes[row, 3].imshow(overlay)

    for col in range(4):
        axes[row, col].axis('off')

# 범례
legend_patches = [
    mpatches.Patch(color='lime',  label='True Positive'),
    mpatches.Patch(color='red',   label='False Positive'),
    mpatches.Patch(color='blue',  label='False Negative'),
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('Figure_6_Segmentation_Results.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: Figure_6_Segmentation_Results.png")

In [ ]:
# ─────────── Figure 7: 전체 검증셋 정량 평가 요약 ───────────
# 보고서 위치: '정량적 성능 평가' 섹션

iou_scores, dice_scores = [], []

for imgs, masks in val_dataset:
    preds = model.predict(imgs, verbose=0)
    preds_bin = (preds > 0.5).astype(np.float32)
    masks_np  = masks.numpy()

    for i in range(len(imgs)):
        p = preds_bin[i].squeeze()
        m = masks_np[i].squeeze()
        intersection = (p * m).sum()
        union = p.sum() + m.sum() - intersection
        iou  = (intersection + 1e-7) / (union + 1e-7)
        dice = (2 * intersection + 1e-7) / (p.sum() + m.sum() + 1e-7)
        iou_scores.append(iou)
        dice_scores.append(dice)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Figure 7: Validation Set Performance Distribution',
             fontsize=13, fontweight='bold')

axes[0].hist(iou_scores,  bins=15, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(np.mean(iou_scores), color='red', linestyle='--',
                label=f'Mean IoU: {np.mean(iou_scores):.3f}')
axes[0].set_title('IoU Distribution')
axes[0].set_xlabel('IoU Score')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(dice_scores, bins=15, color='coral',     edgecolor='white', alpha=0.85)
axes[1].axvline(np.mean(dice_scores), color='navy', linestyle='--',
                label=f'Mean Dice: {np.mean(dice_scores):.3f}')
axes[1].set_title('Dice Coefficient Distribution')
axes[1].set_xlabel('Dice Score')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Figure_7_Performance_Distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: Figure_7_Performance_Distribution.png")
print(f"\n검증셋 최종 성능")
print(f"  Mean IoU  : {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")
print(f"  Mean Dice : {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")

In [ ]:
# ──────────────── 최종 모델 저장 ────────────────
save_path = os.path.join(DRIVE_ROOT, 'final_model')
model.save(save_path)
print(f"모델 저장 완료: {save_path}")

print("\n생성된 Figure 파일 목록:")
for f in sorted(glob.glob('Figure_*.png')):
    print(f"  {f}")

---

## 📋 보고서 작성 가이드 — Figure 배치 위치

| Figure 파일명 | 내용 | 보고서 배치 위치 |
|---|---|---|
| `Figure_1_Data_Samples.png` | 원본 이미지 및 GT 마스크 샘플 | **2장 '데이터'** 섹션 첫 번째 그림 |
| `Figure_2_Augmentation_Examples.png` | 증강 전후 비교 (원본 + 4가지 변형) | **3장 '데이터 증강 기법 선택 근거'** 섹션 |
| `Figure_3_Resolution_Comparison.png` | 128/256/384 해상도별 마스크 보존 비교 | **4장 '입력 해상도 결정 과정 (실험 결과)'** 섹션 |
| `Figure_4_Model_Architecture.png` | U-Net 모델 구조도 | **5장 '모델 구조'** 섹션 |
| `Figure_5_Training_History.png` | Loss / IoU / Dice 학습 곡선 | **6장 '학습 곡선'** 섹션 |
| `Figure_6_Segmentation_Results.png` | 입력·GT·예측·Overlay 대표 결과 (4행) | **7장 '대표 결과 이미지'** 섹션 |
| `Figure_7_Performance_Distribution.png` | 검증셋 IoU/Dice 분포 히스토그램 | **7장 '정량적 성능 평가'** 섹션 |